In [1]:
# 1️Gerekli kütüphaneler
!pip install albumentations opencv-python --quiet

In [2]:
import shutil
import os

folder_path = '/kaggle/working/augmented_dataset'

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print(f"✅ Klasör silindi: {folder_path}")
else:
    print("⚠️ Klasör bulunamadı:", folder_path)


⚠️ Klasör bulunamadı: /kaggle/working/augmented_dataset


In [3]:
import os
import cv2
import shutil
from collections import defaultdict
import albumentations as A
import math

# ============================
# 1️⃣ Dataset yolunu ayarla
# ============================
base_path = '/kaggle/input/surungenbocekdataset/SurungenBocekDataset'
output_path = '/kaggle/working/augmented_dataset'

splits = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
print("Split klasörleri tespit edildi:", splits)

# ============================
# 2️⃣ Sınıf isimleri
# ============================
classes = [
    "Akdeniz Munzevi Orumcegi","Anadolu Sari Akrebi","Kara Akrep","Katil Ari",
    "Yaprak Biti","Kahverengi Kokarca Bocegi","Lahana Tittillari","Patates Bocegi",
    "Misir Kurdu","Misir Yuvarlak Kurdu","Sonbahar Ordu Kurdu","Sirke Sinegi",
    "Kum Yengeci","uc benekli yuzuen yengec","Kirmizi Orumcek","Trips",
    "Mavi Yengec","Kemanci Yengec","baklagil kabarcik bocegi","Camur Yengeci",
    "Pirinc Gal Sinegi","Beyaz Sirtli Bitki Zararlisi"
]

# ============================
# 3️⃣ Augmentation ayarları
# ============================
target_count = 1000   # her sınıf için hedef örnek
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.7),
    A.RandomBrightnessContrast(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5)
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

# ============================
# 4️⃣ Çıktı klasörlerini hazırla
# ============================
if os.path.exists(output_path):
    shutil.rmtree(output_path)
for split in splits:
    os.makedirs(os.path.join(output_path, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_path, split, 'labels'), exist_ok=True)

# ============================
# 5️⃣ Her sınıfın örneklerini say
# ============================
id_count = defaultdict(list)
for split in splits:
    labels_dir = os.path.join(base_path, split, 'labels')
    for f in os.listdir(labels_dir):
        file_path = os.path.join(labels_dir, f)
        with open(file_path, 'r') as file:
            lines = file.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            label_id = int(float(parts[0]))
            id_count[label_id].append((split, f))

# ============================
# 6️⃣ Kopyalama ve augment
# ============================
for label_id, files in id_count.items():
    for split, label_file in files:
        # Resim ve label path
        label_path = os.path.join(base_path, split, 'labels', label_file)
        img_file = label_file.replace('.txt', '.jpg')
        img_path = os.path.join(base_path, split, 'images', img_file)
        if not os.path.exists(img_path):
            img_file = label_file.replace('.txt', '.png')
            img_path = os.path.join(base_path, split, 'images', img_file)
        if not os.path.exists(img_path):
            print("⚠️ Eksik eşleşme bulundu, atlandı:", label_file)
            continue

        # Orijinali kopyala
        shutil.copy2(img_path, os.path.join(output_path, split, 'images', img_file))
        shutil.copy2(label_path, os.path.join(output_path, split, 'labels', label_file))

    # Eksik sınıfları augment et
    if len(files) < target_count and len(files) > 0:
        aug_factor = math.ceil((target_count - len(files)) / len(files))
        for split, label_file in files:
            label_path = os.path.join(base_path, split, 'labels', label_file)
            img_file = label_file.replace('.txt', '.jpg')
            img_path = os.path.join(base_path, split, 'images', img_file)
            if not os.path.exists(img_path):
                img_file = label_file.replace('.txt', '.png')
                img_path = os.path.join(base_path, split, 'images', img_file)
            if not os.path.exists(img_path):
                continue

            with open(label_path, 'r') as f:
                lines = f.readlines()

            bboxes, labels_list = [], []
            image = cv2.imread(img_path)
            if image is None:
                print("⚠️ Görsel okunamadı, atlandı:", img_path)
                continue
            h_img, w_img = image.shape[:2]

            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                lbl_id = int(float(parts[0]))
                xc, yc, w, h = map(float, parts[1:])
                labels_list.append(lbl_id)
                # YOLO -> Pascal (pixel cinsinden)
                x_min = (xc - w / 2) * w_img
                y_min = (yc - h / 2) * h_img
                x_max = (xc + w / 2) * w_img
                y_max = (yc + h / 2) * h_img
                bboxes.append([x_min, y_min, x_max, y_max])

            for i in range(aug_factor):
                augmented = transform(image=image, bboxes=bboxes, class_labels=labels_list)
                aug_img = augmented['image']
                aug_bboxes = augmented['bboxes']
                aug_labels = augmented['class_labels']

                new_img = img_file.replace('.jpg', f'_aug{i}.jpg').replace('.png', f'_aug{i}.png')
                cv2.imwrite(os.path.join(output_path, split, 'images', new_img), aug_img)

                new_label = new_img.replace('.jpg', '.txt').replace('.png', '.txt')
                label_lines = []
                for l, (x_min, y_min, x_max, y_max) in zip(aug_labels, aug_bboxes):
                    # Pascal -> YOLO (normalize edilmiş)
                    xc = ((x_min + x_max) / 2) / w_img
                    yc = ((y_min + y_max) / 2) / h_img
                    w  = (x_max - x_min) / w_img
                    h  = (y_max - y_min) / h_img
                    # clamp [0,1]
                    xc, yc, w, h = max(0, min(1, xc)), max(0, min(1, yc)), max(0, min(1, w)), max(0, min(1, h))
                    label_lines.append(f"{l} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
                with open(os.path.join(output_path, split, 'labels', new_label), 'w') as f:
                    f.writelines(label_lines)

print("✅ Augmentation tamamlandı. Tüm düşük veri sınıfları 1000 örneğe tamamlandı.")


Split klasörleri tespit edildi: ['valid', 'test', 'train']


/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


✅ Augmentation tamamlandı. Tüm düşük veri sınıfları 1000 örneğe tamamlandı.


In [4]:
import os
from collections import Counter

# Çıkış dataset klasörü (augment sonrası)
output_path = '/kaggle/working/augmented_dataset'

# Split klasörleri
splits = ['train', 'valid', 'test']

# Sınıf isimleri
classes = [
    "Akdeniz Munzevi Orumcegi","Anadolu Sari Akrebi","Kara Akrep","Katil Ari",
    "Yaprak Biti","Kahverengi Kokarca Bocegi","Lahana Tittillari","Patates Bocegi",
    "Misir Kurdu","Misir Yuvarlak Kurdu","Sonbahar Ordu Kurdu","Sirke Sinegi",
    "Kum Yengeci","uc benekli yuzuen yengec","Kirmizi Orumcek","Trips",
    "Mavi Yengec","Kemanci Yengec","baklagil kabarcik bocegi","Camur Yengeci",
    "Pirinc Gal Sinegi","Beyaz Sirtli Bitki Zararlisi"
]

# Sayaç
class_counter = Counter()

# Tüm label dosyalarını oku ve say
for split in splits:
    labels_dir = os.path.join(output_path, split, 'labels')
    for f in os.listdir(labels_dir):
        with open(os.path.join(labels_dir, f), 'r') as file:
            lines = file.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                class_id = int(float(parts[0]))
            except:
                continue
            class_counter[class_id] += 1

# ID’ye göre sıralı yazdır
print("📊 Sınıf ID - Sınıf Adı - Toplam Örnek Sayısı\n")
for class_id in sorted(class_counter.keys()):
    class_name = classes[class_id] if class_id < len(classes) else "Unknown"
    count = class_counter[class_id]
    print(f"{class_id:2d} - {class_name:35s} - {count}")


📊 Sınıf ID - Sınıf Adı - Toplam Örnek Sayısı

 0 - Akdeniz Munzevi Orumcegi            - 3439
 1 - Anadolu Sari Akrebi                 - 4610
 2 - Kara Akrep                          - 2893
 3 - Katil Ari                           - 1075
 4 - Yaprak Biti                         - 1376
 5 - Kahverengi Kokarca Bocegi           - 1350
 6 - Lahana Tittillari                   - 900
 7 - Patates Bocegi                      - 1023
 8 - Misir Kurdu                         - 1044
 9 - Misir Yuvarlak Kurdu                - 1038
10 - Sonbahar Ordu Kurdu                 - 1023
11 - Sirke Sinegi                        - 1126
12 - Kum Yengeci                         - 728
13 - uc benekli yuzuen yengec            - 688
14 - Kirmizi Orumcek                     - 1723
15 - Trips                               - 1092
16 - Mavi Yengec                         - 742
17 - Kemanci Yengec                      - 1045
18 - baklagil kabarcik bocegi            - 1035
19 - Camur Yengeci                       - 102

In [5]:
import shutil

# Klasör yolu
folder_path = '/kaggle/working/augmented_dataset'
zip_path = '/kaggle/working/augmented_dataset.zip'

# Klasörü ziple
shutil.make_archive('/kaggle/working/augmented_dataset', 'zip', folder_path)

print("✅ Klasör ziplendi:", zip_path)


✅ Klasör ziplendi: /kaggle/working/augmented_dataset.zip


In [6]:
import os, shutil

# Output klasörünü oluştur
os.makedirs("/kaggle/working/output", exist_ok=True)

# Dosyayı kopyala
shutil.copy("/kaggle/working/augmented_dataset.zip", "/kaggle/working/output/augmented_dataset.zip")


'/kaggle/working/output/augmented_dataset.zip'